## 1. Threshold Logic Unit (TLU)

A TLU is just a computational unit: weighted sum, then threshold. It has no training part built in.

`y_pred = activation_func(z)` <br>

  `where z = wx + b `

  `activation_func = step function`     (or sign function)

Weights and Bias are set by hand, they are not learnable. TLU works perfectly as a logic gate. What it lacks is a way to find its own weights.

In [1]:
import numpy as np

class TLU:
    def __init__(self,w,b):
        self.w = np.array(w,dtype=float)    # converts weight values to numpy arrays of float so that dot product (in w.x + b) works correctly
        self.b = b
    
    def predict(self,X):
        z = np.dot(X,self.w) + self.b       # np.dot(X,self.w) : takes each row of X, multiplies it element by element with w, and sums the products.
        return np.where(z>=0, 1, 0)        # threshold : if z>=0 return 1 else 0  (activation func)

In [2]:
## test TLU for AND 

X = np.array([[0,0],
              [0,1],
              [1,0],
              [1,1]])
w = [1,1]
b = -1.5     # both w and b are handset 

and_gate = TLU(w,b)  # create TLU obj

print(and_gate.predict(X))   # predict


[0 0 0 1]


### Diagrammatic representation of AND Gate TLU

![AND Gate TLU](images/tlu_and_gate_diagram.png)

## 2. Perceptron  (for AND)

A Perceptron is a TLU with a learning rule. It finds its own weights and bias from the data.

`y_pred = activation_func(z)` <br>

  `where z = wx + b `

  `activation_func = step function`     (or sign function)

Weights and Bias start at zero (or small random values) and are **learned**. The perceptron checks its guess against the right answer and fixes itself when it is wrong.

**Perceptron learning rule** (for each sample) <br>

  `error = y_true - y_pred` <br>

  `w = w + lr * error * x` <br>

  `b = b + lr * error`

- `error = 0` : guess is right, nothing changes
- `error = +1` : said 0, should be 1, so `z` goes up 

- `error = -1` : said 1, should be 0, so `z` goes down

`lr` (learning rate) is the step size of each fix. One full pass over the data is an epoch. Training stops when an epoch has 0 errors, or when `epochs` runs out.

A perceptron works only if the data is linearly separable (one straight line can split the classes). AND and OR work. XOR does not, so the errors never reach 0.

In [3]:
class Perceptron:
    def __init__(self, n_features, lr=0.1, epochs=20):   # n_features : number of inputs each sample has
        self.w = np.zeros(n_features)     # n_features=2 gives w = [0., 0.], one weight each for x₁ and x₂
        self.b = 0.0
        self.lr = lr
        self.epochs = epochs

    def activation(self, z):
        return np.where(z >= 0, 1, 0)

    def predict(self, X):
        z = np.dot(X, self.w) + self.b
        return self.activation(z)

    def fit(self, X, y):
        for epoch in range(self.epochs):
            errors = 0
            for xi, target in zip(X, y):     # target is y_true in other words
                pred = self.predict(xi)      # pred is y_pred in other words
                error = target - pred     
                self.w += self.lr * error * xi
                self.b += self.lr * error
                errors += int(error != 0)
            print(f"Epoch {epoch+1}: {errors} errors")
            if errors == 0:   # converged
                break

In [4]:
## test 

X = np.array([[0,0],[0,1],[1,0],[1,1]])
y = np.array([0, 0, 0, 1])

p = Perceptron(n_features=2, lr=1, epochs=20)
p.fit(X, y)
print(p.predict(X))   # [0 0 0 1]
print(p.w, p.b)       # [2. 1.] -3.0

Epoch 1: 2 errors
Epoch 2: 3 errors
Epoch 3: 3 errors
Epoch 4: 2 errors
Epoch 5: 1 errors
Epoch 6: 0 errors
[0 0 0 1]
[2. 1.] -3.0


### Diagrammatic representation of AND Gate Perceptron

![AND Gate Perceptron](images/perceptron_with_learning_loop.png)

## 3. Layered Perceptron (multi-label classifier)

A single layer with several perceptrons just means several outputs, where each output has its own weights and bias, but all of them look at the same inputs.

In [5]:
import numpy as np

class PerceptronLayer:
    def __init__(self, n_features, n_outputs, lr=0.1, epochs=20):
        self.W = np.zeros((n_features, n_outputs))
        self.b = np.zeros(n_outputs)
        self.lr = lr
        self.epochs = epochs

    def activation(self, z):
        return np.where(z >= 0, 1, 0)

    def predict(self, X):
        z = np.dot(X, self.W) + self.b
        return self.activation(z)

    def fit(self, X, Y):
        for epoch in range(self.epochs):
            errors = 0
            for xi, target in zip(X, Y):
                pred = self.predict(xi)          # shape (n_outputs,)
                error = target - pred            # one error per output
                self.W += self.lr * np.outer(xi, error)
                self.b += self.lr * error
                errors += int(np.any(error != 0))
            print(f"Epoch {epoch+1}: {errors} errors")
            if errors == 0:
                break

In [6]:
X = np.array([[0,0],[0,1],[1,0],[1,1]])
Y = np.array([[0,0],     # columns: [AND, OR]
              [0,1],
              [0,1],
              [1,1]])

layer = PerceptronLayer(n_features=2, n_outputs=2, lr=1, epochs=20)
layer.fit(X, Y)
print(layer.predict(X))   # should match Y
print("**")
print(layer.W)
print("**")
print(layer.b)

Epoch 1: 3 errors
Epoch 2: 4 errors
Epoch 3: 4 errors
Epoch 4: 2 errors
Epoch 5: 1 errors
Epoch 6: 0 errors
[[0 0]
 [0 1]
 [0 1]
 [1 1]]
**
[[2. 1.]
 [1. 1.]]
**
[-3. -1.]


### Diagrammatic representation of single layer Perceptron (2 nodes)

![single layer perceptrom](images/single_layer_perceptron_2x2.png)